# Chapter 8: Machine learning for clinical prediction

**Data Analysis with Python for Health Specialists** | 3 hours

---

## 1. When to use ML vs traditional statistics

- **Traditional statistics** (regression, t-tests): "Which risk factors are associated with heart disease, and by how much?" Goal: *inference*.
- **Machine learning**: "Given this patient's data, what is their probability of heart disease?" Goal: *prediction*.

When **NOT** to use ML:
- Small datasets (< 200 patients): ML overfits easily.
- When you need regulatory approval: FDA requires interpretable models.
- When a simple rule works: if "glucose > 126" classifies 90% of diabetics, you don't need a random forest.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_curve, roc_auc_score, RocCurveDisplay)

## 2. Loading clinical data

We use the UCI Heart Disease dataset --- a classic benchmark in clinical ML.

In [ ]:
# UCI Heart Disease dataset
url = ("https://raw.githubusercontent.com/raphaelfontenelle/"
       "heart-disease-uci/main/heart.csv")
heart = pd.read_csv(url)
print(f"Shape: {heart.shape}")
print(f"Heart disease prevalence: {heart['target'].mean():.1%}")
heart.head()

In [ ]:
# Column descriptions
column_info = {
    "age": "Age in years",
    "sex": "1 = male, 0 = female",
    "cp": "Chest pain type (0-3)",
    "trestbps": "Resting blood pressure (mmHg)",
    "chol": "Serum cholesterol (mg/dL)",
    "fbs": "Fasting blood sugar > 120 mg/dL (1 = true)",
    "restecg": "Resting ECG results (0-2)",
    "thalach": "Maximum heart rate achieved",
    "exang": "Exercise-induced angina (1 = yes)",
    "oldpeak": "ST depression induced by exercise",
    "slope": "Slope of peak exercise ST segment",
    "ca": "Number of major vessels colored by fluoroscopy (0-3)",
    "thal": "Thalassemia (1 = normal, 2 = fixed defect, 3 = reversible)",
    "target": "Heart disease (1 = yes, 0 = no)"
}
for col, desc in column_info.items():
    print(f"  {col:12s} : {desc}")

## 3. Train/test split and cross-validation

In [ ]:
# Define features and target
feature_cols = ["age", "sex", "cp", "trestbps", "chol", "fbs",
                "restecg", "thalach", "exang", "oldpeak", "slope", "ca", "thal"]
X = heart[feature_cols]
y = heart["target"]

# 70/30 train/test split (stratified to preserve class balance)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
print(f"Training set: {X_train.shape[0]} patients")
print(f"Test set:     {X_test.shape[0]} patients")
print(f"Training prevalence: {y_train.mean():.1%}")
print(f"Test prevalence:     {y_test.mean():.1%}")

In [ ]:
# 5-fold cross-validation with logistic regression
lr = LogisticRegression(max_iter=1000, random_state=42)
cv_scores = cross_val_score(lr, X, y, cv=5, scoring="accuracy")
print(f"CV accuracy: {cv_scores.mean():.3f} +/- {cv_scores.std():.3f}")
print(f"Per-fold: {cv_scores.round(3)}")

## 4. Decision trees

A decision tree splits patients using yes/no questions, forming a flowchart clinicians can follow.

In [ ]:
# Fit a decision tree
dt = DecisionTreeClassifier(max_depth=4, random_state=42)
dt.fit(X_train, y_train)

# Evaluate
y_pred_dt = dt.predict(X_test)
print("Decision Tree Results:")
print(classification_report(y_test, y_pred_dt,
                            target_names=["No Disease", "Disease"]))

In [ ]:
# Visualize the tree
fig, ax = plt.subplots(figsize=(20, 10))
plot_tree(dt, feature_names=feature_cols,
          class_names=["No Disease", "Disease"],
          filled=True, rounded=True, fontsize=9, ax=ax)
plt.title("Decision tree for heart disease prediction")
plt.tight_layout()
plt.show()

### The danger of deep trees

In [ ]:
# Overfit tree (no depth limit)
dt_overfit = DecisionTreeClassifier(random_state=42)  # no max_depth
dt_overfit.fit(X_train, y_train)

print(f"Training accuracy (overfit): {dt_overfit.score(X_train, y_train):.3f}")
print(f"Test accuracy (overfit):     {dt_overfit.score(X_test, y_test):.3f}")
print(f"Training accuracy (depth=4): {dt.score(X_train, y_train):.3f}")
print(f"Test accuracy (depth=4):     {dt.score(X_test, y_test):.3f}")

## 5. Random forests

A random forest combines hundreds of decision trees, each trained on a random subset of data and features.

In [ ]:
# Fit a random forest
rf = RandomForestClassifier(n_estimators=200, max_depth=6,
                            random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
print("Random Forest Results:")
print(classification_report(y_test, y_pred_rf,
                            target_names=["No Disease", "Disease"]))

In [ ]:
# Cross-validation comparison
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree (depth=4)": DecisionTreeClassifier(max_depth=4, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, max_depth=6,
                                            random_state=42, n_jobs=-1)
}

print("5-fold CV accuracy:")
for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=5, scoring="accuracy")
    print(f"  {name:30s}: {scores.mean():.3f} +/- {scores.std():.3f}")

## 6. ROC curves and AUC

- AUC = 0.5: no better than coin flip
- AUC = 0.7-0.8: acceptable
- AUC = 0.8-0.9: excellent
- AUC > 0.9: outstanding (rare in clinical practice)

In [ ]:
# Get predicted probabilities
lr.fit(X_train, y_train)
y_prob_lr = lr.predict_proba(X_test)[:, 1]
y_prob_rf = rf.predict_proba(X_test)[:, 1]

# Compute ROC curves
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_prob_lr)
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_prob_rf)
auc_lr = roc_auc_score(y_test, y_prob_lr)
auc_rf = roc_auc_score(y_test, y_prob_rf)

# Plot
fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr_lr, tpr_lr, label=f"Logistic Regression (AUC = {auc_lr:.3f})",
        linewidth=2)
ax.plot(fpr_rf, tpr_rf, label=f"Random Forest (AUC = {auc_rf:.3f})",
        linewidth=2)
ax.plot([0, 1], [0, 1], "k--", alpha=0.5, label="Random (AUC = 0.500)")
ax.set_xlabel("False Positive Rate (1 - Specificity)")
ax.set_ylabel("True Positive Rate (Sensitivity)")
ax.set_title("ROC curves: Heart disease prediction")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

## 7. Feature importance

**Question:** Which risk factors matter most for predicting heart disease?

In [ ]:
# Random forest feature importance
importances = rf.feature_importances_
importance_df = pd.DataFrame({
    "Feature": feature_cols,
    "Importance": importances
}).sort_values("Importance", ascending=True)

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(importance_df["Feature"], importance_df["Importance"], color="steelblue")
ax.set_xlabel("Feature importance (Gini)")
ax.set_title("Which risk factors predict heart disease?")
plt.tight_layout()
plt.show()

In [ ]:
# Top 5 features
print("Top 5 most important features:")
for _, row in importance_df.tail(5).iloc[::-1].iterrows():
    print(f"  {row['Feature']:12s}: {row['Importance']:.3f}")

## 8. Calibration

A well-calibrated model's predicted probabilities match observed outcomes.

In [ ]:
from sklearn.calibration import calibration_curve

# Calibration plot for random forest
prob_true, prob_pred = calibration_curve(y_test, y_prob_rf, n_bins=8)

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(prob_pred, prob_true, "o-", linewidth=2, label="Random Forest")
ax.plot([0, 1], [0, 1], "k--", label="Perfectly calibrated")
ax.set_xlabel("Mean predicted probability")
ax.set_ylabel("Observed proportion")
ax.set_title("Calibration plot")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Brier score: lower is better (0 = perfect)
from sklearn.metrics import brier_score_loss

brier_lr = brier_score_loss(y_test, y_prob_lr)
brier_rf = brier_score_loss(y_test, y_prob_rf)
print(f"Brier score (Logistic Regression): {brier_lr:.4f}")
print(f"Brier score (Random Forest):       {brier_rf:.4f}")

## 9. Overfitting: the danger of small clinical datasets

In [ ]:
# Demonstrate overfitting with varying dataset sizes
train_sizes = [30, 50, 100, 150, 200, len(X_train)]
results = []

for size in train_sizes:
    if size > len(X_train):
        continue
    X_sub = X_train.iloc[:size]
    y_sub = y_train.iloc[:size]

    rf_temp = RandomForestClassifier(n_estimators=100, random_state=42)
    rf_temp.fit(X_sub, y_sub)

    train_acc = rf_temp.score(X_sub, y_sub)
    test_acc = rf_temp.score(X_test, y_test)
    results.append({"n_train": size, "train_acc": train_acc, "test_acc": test_acc})

results_df = pd.DataFrame(results)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(results_df["n_train"], results_df["train_acc"], "o-",
        label="Training accuracy", linewidth=2)
ax.plot(results_df["n_train"], results_df["test_acc"], "s-",
        label="Test accuracy", linewidth=2)
ax.set_xlabel("Number of training patients")
ax.set_ylabel("Accuracy")
ax.set_title("Learning curve: more data reduces overfitting")
ax.legend()
ax.set_ylim(0.5, 1.05)
plt.tight_layout()
plt.show()

## 10. Complete pipeline and fairness audit

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Full pipeline
pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", RandomForestClassifier(n_estimators=200, max_depth=6,
                                          random_state=42))
])

# Cross-validate the full pipeline
cv_scores = cross_val_score(pipeline, X, y, cv=5, scoring="roc_auc")
print(f"Pipeline CV AUC: {cv_scores.mean():.3f} +/- {cv_scores.std():.3f}")

# Final fit and evaluation
pipeline.fit(X_train, y_train)
y_prob_final = pipeline.predict_proba(X_test)[:, 1]
y_pred_final = pipeline.predict(X_test)

print(f"\nFinal test AUC: {roc_auc_score(y_test, y_prob_final):.3f}")
print(f"\n{classification_report(y_test, y_pred_final, target_names=['No Disease', 'Disease'])}")

In [ ]:
# Check model performance by sex (fairness audit)
for sex_val, sex_label in [(1, "Male"), (0, "Female")]:
    mask = X_test["sex"] == sex_val
    if mask.sum() < 10:
        print(f"{sex_label}: too few samples ({mask.sum()})")
        continue

    auc_sub = roc_auc_score(y_test[mask], y_prob_final[mask])
    sens = (y_pred_final[mask] == 1)[y_test[mask] == 1].mean()
    spec = (y_pred_final[mask] == 0)[y_test[mask] == 0].mean()

    print(f"{sex_label:8s}: AUC = {auc_sub:.3f}, "
          f"Sensitivity = {sens:.3f}, Specificity = {spec:.3f}, "
          f"n = {mask.sum()}")

## Exercises

Using the UCI Heart Disease dataset:

1. **Baseline.** Fit logistic regression using all features. Report CV accuracy and AUC.
2. **Decision tree.** Fit with max_depth=3. Visualize. Can a clinician follow this at the bedside?
3. **Random forest.** Fit with 200 trees. Compare AUC to logistic regression. Is the improvement worth the loss of interpretability?
4. **Feature selection.** Train using only the top 5 features. How does performance compare to all 13?
5. **ROC comparison.** Plot ROC curves for all three models on the same figure.
6. **Threshold analysis.** Plot sensitivity and specificity vs classification threshold (0 to 1). At what threshold are they equal?
7. **Calibration.** Plot calibration curves for logistic regression and random forest.
8. **Fairness audit.** Compare AUC, sensitivity, specificity between male and female patients.
9. **Clinical deployment.** Write a paragraph describing how you would validate this model before hospital use.

In [ ]:
# Exercise 1: your code here


In [ ]:
# Exercise 2: your code here


In [ ]:
# Exercise 3: your code here


In [ ]:
# Exercise 4: your code here


In [ ]:
# Exercise 5: your code here


In [ ]:
# Exercise 6: your code here


In [ ]:
# Exercise 7: your code here


In [ ]:
# Exercise 8: your code here


# Exercise 9: your written answer here
